# Trabajo Práctico: Aprendizaje Supervisado

## Presentación del dataset

Se realizó un estudio extenso acerca de cómo los hábitos de los estudiantes, su estado psicológico y su entrega de trabajos se relacionan con la nota que obtuvieron
en el examen final. El dataset incluye:
- **Características académicas**: Como las horas de estudio, el porcentaje de entrega de trabajos, de asistencia, etc.
- **Características psicológicas**: Captura factores como el nivel de estrés, la motivación, la concentración, etc.
- **Caraceterísticas de estilo de vida digital**: Representa hábitos digitales como las horas que pasan en redes sociales.
- **Características de salud**: Duración de sueño, actividad física, etc.

Las variables objetivo son dos:
- `final_exam_score`: El resultado que obtuvieron en su último examen.
- `performance_category`: La categoría de rendimiento a la que pertenecen.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('student_performance.csv')

# primeras filas para entender la estructura
print('Primeras filas del dataset:')
print(df.head())

# dimensiones: filas y columnas
print(f'\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')

# resumen de tipos de datos y valores nulos
print('\nInformación general:')
df.info()

# estadísticas descriptivas de las columnas numéricas
print('\nEstadísticas descriptivas:')
print(df.describe())

In [ ]:
# valores únicos de columnas categóricas
print('Columnas categóricas y sus valores únicos:')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].unique().tolist()}')

# distribución de la variable objetivo
df['final_exam_score'].hist(bins=30, color='steelblue', edgecolor='white')
plt.title('Distribución de final_exam_score')
plt.xlabel('Nota final')
plt.ylabel('Cantidad de estudiantes')
plt.tight_layout()
plt.show()

## Actividades

1. Suponga que tratamos de predecir la nota que los estudiantes obtuvieron en su examen final.
    - a) ¿De qué tipo tarea se trata? 
    - b) ¿Qué algoritmo de aprendizaje automático podríamos usar para resolverla?

a) Como la nota es un valor numérico continuo (entre 0 y 100), la tarea es de **Regresión**. Además, como contamos con las notas reales de cada alumno para entrenar al modelo, es un problema de aprendizaje **supervisado**.

b) El algoritmo más directo y común para este tipo de problema es la **Regresión Lineal**, que busca trazar una línea matemática que pase lo más cerca posible de todos los datos, relacionando las características de cada estudiante con su nota final.

2. Elige entre **5 y 10 columnas** que te parezcan más relevantes para predecir la nota que los estudiantes obtuvieron en su examen final. 
    **Requerimiento:** Escoge al menos una columna categórica. ¿Podemos aplicar regresión directamente con esa variable categórica?
    - a) Aplica las técnicas de procesado que creas necesarias para estas columnas categóricas. 
    - b) ¿Podemos aplicar regresión cuando hay datos nulos? Si no, aplica técnicas para manejar datos nulos y justifícalas.

In [ ]:
df.isnull().sum()

In [ ]:
columnas_elegidas = [
    'study_hours_per_day',
    'assignment_completion_rate',
    'motivation_level',
    'focus_score',
    'procrastination_index',
    'learning_style'  # columna categórica
]

# separamos los datos de entrada (X) y lo que queremos predecir (y)
X = df[columnas_elegidas].copy()
Y = df['final_exam_score']

In [ ]:
# focus_score tiene 60 valores nulos, los rellenamos con el promedio de todos los estudiantes
promedio_foco = X['focus_score'].mean()
X['focus_score'] = X['focus_score'].fillna(promedio_foco)

print(f'Nulos en focus_score rellenados con la media: {promedio_foco:.2f}')
print(f'Nulos restantes: {X.isnull().sum().sum()}')

In [ ]:
# No podemos usar texto directamente en regresión lineal
# convertimos learning_style a columnas numéricas con get_dummies (One-Hot Encoding)
X = pd.get_dummies(X, columns=['learning_style'], drop_first=True)

X.head()

No podemos aplicar regresión directamente con variables categóricas porque el algoritmo trabaja con operaciones matemáticas y no puede sumar o restar texto. Para resolverlo usamos **get_dummies** (One-Hot Encoding), que convierte cada categoría en una columna de 0s y 1s. Con `drop_first=True` eliminamos una columna para evitar redundancia.

b) Tampoco podemos entrenar el modelo con valores nulos, ya que rompen los cálculos internos. `focus_score` tenía 60 nulos, por lo que los rellenamos con el **promedio** de esa columna. Usamos la media porque los datos no muestran valores extremos que puedan distorsionarla.

3. Entrena un modelo que se ajuste a los datos y que prediga la nota que los estudiantes obtuvieron en su examen final. Sólo considera las características que mencionaste en el punto 2).
    - a) Calcula el error cuadrático medio y la raíz del error cuadrático medio. ¿Qué interpretación tiene cada métrica?
    - b) Calcula el coeficiente de determinación ($R^2$). 
    - c) ¿Qué nos indica el coeficiente en este caso? ¿Existe una relación lineal?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

In [ ]:
modelo_regresion = LinearRegression()
modelo_regresion.fit(X, Y)

predicciones = modelo_regresion.predict(X)

# comparamos las notas reales (Y) con las notas que predijo el modelo (predicciones)

# error cuadrático medio
mse = mean_squared_error(Y, predicciones)

# raíz del error cuadrático medio
rmse = root_mean_squared_error(Y, predicciones)

# coeficiente de determinación
r2 = r2_score(Y, predicciones)

In [ ]:
print('Resultados del modelo')
print(f'MSE (Error cuadrático medio): {mse:.2f}')
print(f'RMSE (Raíz del error cuadrático medio): {rmse:.2f} puntos')
print(f'R² (Coeficiente de determinación): {r2:.4f}')

a)
**MSE**: representa el promedio de los errores del modelo elevados al cuadrado. Su interpretación directa es difícil porque está en una unidad irreal ("puntos al cuadrado"), pero sirve matemáticamente para penalizar con más fuerza los errores muy grandes.

**RMSE**: nos indica cuántos puntos se equivoca el modelo en promedio cada vez que intenta predecir la nota de un estudiante. Al estar en las mismas unidades que la nota, es mucho más fácil de interpretar.

b)
El cálculo nos devuelve el R² del modelo con las 5 características seleccionadas.

c)
El R² nos indica qué porcentaje de la variabilidad en las notas finales logran explicar las variables que elegimos. Un valor cerca del 0 significa que la relación lineal es débil, y que una parte importante de la nota depende de otros factores que no incluimos en este primer modelo.

Sí existe una relación lineal, pero es moderada. El modelo es un punto de partida válido, aunque hay margen para mejorarlo con mejores variables.

4. La propiedad `coef_` del modelo entrenado nos indica los **coeficientes** que el modelo ajustado le da a cada característica. 
    - a) ¿Cuál es la característica más importante según el coeficiente que se le asigna? 
    - b) ¿Cómo interpretas que uno de esos coeficientes sea negativo?

In [ ]:
tabla_coeficientes = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': modelo_regresion.coef_
})

In [ ]:
tabla_coeficientes = tabla_coeficientes.sort_values(by='Coeficiente', key=abs, ascending=False)

print(' PESO DE CADA CARACTERÍSTICA ')
print(tabla_coeficientes)

a) La característica más importante es la que tiene el coeficiente de mayor valor absoluto. En este caso `study_hours_per_day` encabeza la tabla: por cada hora extra de estudio diaria, el modelo predice que la nota sube aproximadamente ese número de puntos.

b)
Un coeficiente negativo indica una relación inversa entre esa característica y la nota del examen. Significa que si esa variable aumenta, la nota predicha baja.
- `procrastination_index` tiene coeficiente negativo: por cada punto extra en el índice de procrastinación, la nota final cae. Tiene todo el sentido: cuanto más posterga un estudiante, peor le va.

Por el contrario, los coeficientes positivos suman: más horas de estudio o mayor motivación se traducen en mejor nota predicha.

5. Prueba con otras selecciones de características. 
    - a) Puedes agregar, quitar, o reemplazar las que hayas estudiado, o puedes seleccionar un grupo totalmente diferente (siempre un máximo de 10). 
    - b) Ajusta otro modelo de regresión a los datos seleccionados. 
    - c) Calcula las métricas (MSE, RMSE y R²) para este nuevo modelo y compáralas con las que obtuviste antes. ¿Qué conjunto parece explicar mejor la nota 
    de los estudiantes? 

**Pista**: Ten en cuenta los coeficientes del punto 4). Además, recuerda que dos características que son **codependientes** tienden a empeorar el rendimiento de un modelo de regresión. 

In [ ]:
nuevas_columnas = [
    'study_hours_per_day',
    'attendance_percentage',
    'revision_efficiency',
    'consistency_score',
    'burnout_risk',
    'social_media_hours',
    'mental_state'  # nueva categórica
]

X_nuevo = df[nuevas_columnas].copy()
y = df['final_exam_score']

In [ ]:
# social_media_hours tiene 60 nulos, los rellenamos con el promedio
promedio_redes = X_nuevo['social_media_hours'].mean()
X_nuevo['social_media_hours'] = X_nuevo['social_media_hours'].fillna(promedio_redes)

X_nuevo = pd.get_dummies(X_nuevo, columns=['mental_state'], drop_first=True)

In [ ]:
modelo_nuevo = LinearRegression()
modelo_nuevo.fit(X_nuevo, y)
predicciones_nuevas = modelo_nuevo.predict(X_nuevo)

In [ ]:
mse_nuevo = mean_squared_error(y, predicciones_nuevas)
rmse_nuevo = root_mean_squared_error(y, predicciones_nuevas)
r2_nuevo = r2_score(y, predicciones_nuevas)

In [ ]:
print('RESULTADOS DEL NUEVO MODELO (EXPERIMENTO)')
print(f'Nuevo MSE: {mse_nuevo:.2f}')
print(f'Nuevo RMSE: {rmse_nuevo:.2f} puntos')
print(f'Nuevo R²: {r2_nuevo:.4f}')

Para el nuevo modelo elegí variables que evitan la codependencia: `revision_efficiency` y `consistency_score` están relacionadas con el esfuerzo pero miden aspectos distintos, y reemplacé `motivation_level` y `focus_score` (que tienden a moverse juntos) para reducir la multicolinealidad.

Al comparar los dos modelos, el nuevo R² refleja si esta selección captura mejor la variabilidad de las notas. En general, incluir variables como `burnout_risk` y `consistency_score`, que el propio modelo anterior señalaba como relevantes por su coeficiente, debería mejorar la capacidad de predicción respecto al primer intento.

**Ejercicio extra:** Para los dos modelos que creaste, divide el dataset en un conjunto de test y otro de entrenamiento. 
    - a) Entrena nuevamente los dos modelos, solamente con el conjunto de entrenamiento.
    - b) Repite el cálculo de las métricas, pero esta vez considerando sólo el conjunto de test.
    - c) Compara los resultados utilizando la partición test/entrenamiento y utilizando el conjunto total como entrenamiento.
    - d) ¿Por qué la división entre test y entrenamiento es mejor para medir el rendimiento de un modelo? **Pista**: Si el parcial de una materia fuera exactamente igual al modelo de examen, ¿podríamos decir que el parcial sirve para medir cuánto sabe un alumno?

In [ ]:
from sklearn.model_selection import train_test_split

# a) división 80% entrenamiento - 20% test para ambos modelos
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_nuevo_train, X_nuevo_test, y_nuevo_train, y_nuevo_test = train_test_split(X_nuevo, y, test_size=0.2, random_state=42)

# entrenamos solo con el conjunto de entrenamiento
modelo1_train = LinearRegression()
modelo1_train.fit(X_train, y_train)

modelo2_train = LinearRegression()
modelo2_train.fit(X_nuevo_train, y_nuevo_train)

In [ ]:
# b) predicciones y métricas solo sobre el conjunto de test
pred_test1 = modelo1_train.predict(X_test)
rmse_test1 = root_mean_squared_error(y_test, pred_test1)
r2_test1 = r2_score(y_test, pred_test1)

pred_test2 = modelo2_train.predict(X_nuevo_test)
rmse_test2 = root_mean_squared_error(y_nuevo_test, pred_test2)
r2_test2 = r2_score(y_nuevo_test, pred_test2)

# c) comparación: dataset completo vs partición test
print('Comparación de resultados:')
print(f'{"":20} {"Modelo 1":>15} {"Modelo 2":>15}')
print('-' * 52)
print(f'{"RMSE (completo)":20} {rmse:>15.2f} {rmse_nuevo:>15.2f}')
print(f'{"RMSE (test)":20} {rmse_test1:>15.2f} {rmse_test2:>15.2f}')
print(f'{"R² (completo)":20} {r2:>15.4f} {r2_nuevo:>15.4f}')
print(f'{"R² (test)":20} {r2_test1:>15.4f} {r2_test2:>15.4f}')

c) Al evaluar con el dataset completo, las métricas suelen verse un poco mejores que al evaluar solo con el conjunto de test, porque el modelo ya "vio" esos datos durante el entrenamiento.

d) Si el parcial de una materia fuera exactamente igual al modelo de examen, el alumno podría memorizarlo sin entender el contenido y sacar 10 igual. Lo mismo le pasa a un modelo de ML que se evalúa con los mismos datos con los que aprendió: las métricas parecen buenas, pero en realidad el modelo solo **memorizó** esos datos (a esto se le llama **overfitting**).

La división train/test simula lo que pasa en la realidad: el modelo tiene que predecir datos que nunca vio. Un modelo útil es el que generaliza bien, y la única forma de verificar eso es evaluarlo con datos nuevos.